## Домашнее задание №1

Посчитать количество рейтингов больше 4 для каждого фильма и вывести фильмы в порядке убывания количества этих оценок.

In [1]:
import os
from collections import defaultdict, namedtuple
from functools import reduce
from joblib import Parallel, delayed

In [2]:
with open(os.path.join("..", "data", "user_ratedmovies.dat"), "rb") as f:
    data = f.readlines()
headers = data[0].decode().split("\t")[:3]
data = [row.decode().split("\t")[:3] for row in data[1:]]

In [3]:
headers

['userID', 'movieID', 'rating']

In [4]:
data[0]

['75', '3', '1']

In [5]:
len(data)

855598

Пишем `map`, `shuffle` и `reduce` + параллелим вычисления. Лучше задавать `batch_size` при распараллеливании, либо даже заранее всё разбить на батчи, будет быстрее.

Также посмотрите на то, нет ли перекоса в данных после `shuffle`, можете попробовать использовать остаток от деления не простого hash, а ввести какую-то функию.

In [6]:
PARTITION_SIZE = 1024

partitions = [data[i:i+PARTITION_SIZE] for i in range(0, len(data), PARTITION_SIZE)]

In [7]:
Row = namedtuple("Row", ["movie_id", "rating"])


def map_shuffle(partition: list[list[str]], n_nodes: int) -> defaultdict[int, list[Row]]:
    return shuffle_rating(mapper_rating(partition), n_nodes)


def mapper_rating(partition: list[list[str]]) -> list[Row]:
    return [Row(row[1], float(row[2])) for row in partition]


def shuffle_rating(mapper_result: list[Row], n_nodes: int = 5) -> defaultdict[int, list[Row]]:
    shuffled_data = [(hash(row.movie_id) % n_nodes, row) for row in mapper_result]
    return create_chunks(shuffled_data)


def create_chunks(shuffled_data: list[tuple[int, Row]]) -> defaultdict[int, list[Row]]:
    result = defaultdict(list)
    for idx, data in shuffled_data:
        result[idx].append(data)
    return result

In [8]:
N_NODES = 5


with Parallel(n_jobs=4, verbose=10) as parallel:
    answer1 = parallel(delayed(map_shuffle)(partition, N_NODES) for partition in partitions)

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:    0.5s
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:    0.6s
[Parallel(n_jobs=4)]: Batch computation too fast (0.19705408830883478s.) Setting batch_size=2.
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:    0.6s
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:    0.6s
[Parallel(n_jobs=4)]: Batch computation too fast (0.034793853759765625s.) Setting batch_size=4.
[Parallel(n_jobs=4)]: Done  44 tasks      | elapsed:    0.7s
[Parallel(n_jobs=4)]: Batch computation too fast (0.0665740966796875s.) Setting batch_size=8.
[Parallel(n_jobs=4)]: Done  88 tasks      | elapsed:    1.0s
[Parallel(n_jobs=4)]: Done 176 tasks      | elapsed:    1.4s
[Parallel(n_jobs=4)]: Done 264 tasks      | elapsed:    1.6s
[Parallel(n_jobs=4)]: Batch computation too fast (0.19357569084365883s.) Setting batch_size=16.
[Parallel(n_jobs=4)]: Done 376 tasks      | elapsed:    

In [9]:
shuffle_stage = {i:[] for i in range(N_NODES)}
for partition in answer1:
    for idx, rows in partition.items():
        shuffle_stage[idx].extend(rows)

In [10]:
for idx, rows in shuffle_stage.items():
    print(f"{idx}: number of rows = {len(rows)}")

0: number of rows = 172899
1: number of rows = 174699
2: number of rows = 174078
3: number of rows = 165173
4: number of rows = 168749


In [11]:
def reduce_rating(rows: list[Row]) -> list[tuple[str, float, int]]:
    ratings = defaultdict(float)
    counter = defaultdict(int)
    for row in rows:
        ratings[row.movie_id] += row.rating
        counter[row.movie_id] += 1
    return [
        (movie_id, ratings[movie_id] / counter[movie_id], counter[movie_id])
        for movie_id in ratings
    ]

In [12]:
with Parallel(n_jobs=4, verbose=10) as parallel:
    answer2 = parallel(delayed(reduce_rating)(partition) for partition in shuffle_stage.values())

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   2 out of   5 | elapsed:    1.2s remaining:    1.9s
[Parallel(n_jobs=4)]: Done   3 out of   5 | elapsed:    1.6s remaining:    1.0s
[Parallel(n_jobs=4)]: Done   5 out of   5 | elapsed:    2.3s finished


После `reduce` всё можно собрать в одном цикле, считаем, что данные переслали после на 1 машину и агрегируем.

In [13]:
result = []
for partition in answer2:
    result.extend(partition)

In [14]:
for movie_id, avg_rating, count in sorted(
    filter(lambda x: x[1] > 4.0, result), key=lambda x: x[2], reverse=True
):
    print(f"{movie_id=}: {avg_rating=:.3}, {count=}")

movie_id='2858': avg_rating=4.11, count=1174
movie_id='32': avg_rating=4.01, count=923
movie_id='296': avg_rating=4.23, count=889
movie_id='2571': avg_rating=4.18, count=877
movie_id='4993': avg_rating=4.09, count=845
movie_id='593': avg_rating=4.05, count=830
movie_id='5952': avg_rating=4.03, count=808
movie_id='7153': avg_rating=4.1, count=777
movie_id='1198': avg_rating=4.12, count=758
movie_id='4226': avg_rating=4.19, count=754
movie_id='318': avg_rating=4.35, count=740
movie_id='1197': avg_rating=4.1, count=685
movie_id='1222': avg_rating=4.04, count=682
movie_id='1221': avg_rating=4.22, count=675
movie_id='260': avg_rating=4.08, count=652
movie_id='296': avg_rating=4.24, count=648
movie_id='1080': avg_rating=4.0, count=648
movie_id='7361': avg_rating=4.12, count=623
movie_id='1196': avg_rating=4.14, count=621
movie_id='593': avg_rating=4.05, count=601
movie_id='858': avg_rating=4.34, count=598
movie_id='1136': avg_rating=4.1, count=583
movie_id='912': avg_rating=4.21, count=556
m